In [5]:
import os
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score, d2_absolute_error_score
import numpy as np
import pandas as pd

# Загрузка необходимых ресурсов
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

# Путь к данным
train_path = '../aclImdb/train'

# Функция для чтения данных из файлов
def read_data(folder, label, base_path):
    data = []
    folder_path = os.path.join(base_path, folder)
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            # Чтение текста из файла
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                text = file.read()
            
            # Извлечение target из названия файла
            target = filename.split('_')[1].split('.')[0]
            
            # Добавление данных в список
            data.append({
                'text': text,
                'positive': label,
                'target': int(target)
            })
    
    return data



[nltk_data] Downloading package stopwords to /home/kolya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/kolya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/kolya/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [6]:

# Чтение данных из папок pos и neg
pos_data = read_data('pos', 1, train_path)
neg_data = read_data('neg', 0, train_path)


test_path = '../aclImdb/test'
pos_data_test = read_data('pos', 1, test_path)
neg_data_test = read_data('neg', 0, test_path)


# Объединение данных
all_data = pos_data + neg_data
test_data = pos_data_test + neg_data_test

# Создание DataFrame
df = pd.DataFrame(all_data)
df_test = pd.DataFrame(test_data)


# Вывод первых строк для проверки
print(df.head())

                                                text  positive  target
0  Another demonstration of Kurosawa's genius, hi...         1       8
1  A new side to the story of Victoria and Albert...         1       8
2  Whoever says pokemon is stupid can die. This m...         1      10
3  With Iphigenia, Mikhali Cacoyannis is perhaps ...         1      10
4  I have to start saying it has been a long time...         1       7


In [7]:
# Загрузка стоп-слов
stop_words = set(stopwords.words('english'))

# Инициализация лемматизатора
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    #1. Приведение к нижнему регистру
    text = text.lower()
    
    # 2. Удаление HTML-тегов
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Удаление пунктуации
    text = re.sub(r'_+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    #text = re.sub(r'\d+', '', text)
    
    # 4. Токенизация
    tokens = text.split(' ')
    
    # 5. Удаление стоп-слов
    tokens = [word for word in tokens if word not in stop_words]
    
    # 6. Лемматизация
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # 7. Объединение токенов обратно в строку
    text = ' '.join(tokens)
    
    return text

In [8]:
df.text = df.text.apply(preprocess_text)
df_test.text = df_test.text.apply(preprocess_text)

In [9]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(df.text)
print(vectorizer.get_feature_names_out()[:5])
print(X_train.shape)

['00' '000' '0000000000001' '000001' '0001']
(25000, 133369)


In [10]:
X_test = vectorizer.transform(df_test.text)
print(X_test.shape)

(25000, 133369)


In [11]:
y_train = df.target
y_test = df_test.target
y_train_clf = df.positive
y_test_clf = df_test.positive

In [12]:
# KFold с 5 разбиениями
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Функция для проведения валидации
def validate_model(model, X, y, kf, metric):
    mae_scores = []
    
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Обучение модели
        model.fit(X_train, y_train)
        
        # Предсказание на тестовом наборе
        y_pred = model.predict(X_test)
        
        mae = metric(y_test, y_pred)
        mae_scores.append(mae)
        
    return mae_scores


In [13]:
# Импорт необходимых библиотек
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


# Словарь моделей классификации
models_classification = {
    'Logistic Regression': LogisticRegression(),
    'Ridge classifier': RidgeClassifier(), 
    'Ridge alpha = 3': RidgeClassifier(5),
    #'Support Vector Classifier': SVC(max_iter = 1000),
    'KNN': KNeighborsClassifier(),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=50, max_depth=10)
}


In [14]:
for model_name, model in models_classification.items():
    scores = validate_model(model, X_train, y_train_clf, kf, accuracy_score)
    print(f"{model_name}: Точность на каждом разбиении: {scores}")
    print(f"{model_name}: Средняя точность: {np.mean(scores)}\n")

Logistic Regression: Точность на каждом разбиении: [0.8838, 0.885, 0.88, 0.8916, 0.8818]
Logistic Regression: Средняя точность: 0.88444

Ridge classifier: Точность на каждом разбиении: [0.8934, 0.886, 0.8916, 0.8986, 0.8846]
Ridge classifier: Средняя точность: 0.8908400000000001

Ridge alpha = 3: Точность на каждом разбиении: [0.8858, 0.8878, 0.8834, 0.8928, 0.8856]
Ridge alpha = 3: Средняя точность: 0.8870800000000001

KNN: Точность на каждом разбиении: [0.775, 0.7808, 0.7804, 0.777, 0.781]
KNN: Средняя точность: 0.77884

Random Forest Classifier: Точность на каждом разбиении: [0.7884, 0.799, 0.7972, 0.7878, 0.7974]
Random Forest Classifier: Средняя точность: 0.79396



In [15]:
def test_error(model, model_name, X_test, y_test, X_train, y_train, metric):
    model.fit(X_train, y_train)
    print(f'{model_name}: метрика на train данных: {metric(model.predict(X_train), y_train):.4f}') 
    test_pred = model.predict(X_test)
    f = lambda x: round(x)
    rounded_pred = list(map(f, test_pred))
    print(f'{model_name}: метрика на тестовых данных: {metric(test_pred, y_test):.4f}') 
    print(f'{model_name}: метрика на округлённых тестовых данных: {metric(rounded_pred, y_test):.4f}') 
    print()

In [16]:
for model_name, model in models_classification.items():
    test_error(model, model_name, X_test, y_test_clf,X_train, y_train_clf, accuracy_score)

Logistic Regression: метрика на train данных: 0.9358
Logistic Regression: метрика на тестовых данных: 0.8798
Logistic Regression: метрика на округлённых тестовых данных: 0.8798

Ridge classifier: метрика на train данных: 0.9854
Ridge classifier: метрика на тестовых данных: 0.8680
Ridge classifier: метрика на округлённых тестовых данных: 0.8680

Ridge alpha = 3: метрика на train данных: 0.9440
Ridge alpha = 3: метрика на тестовых данных: 0.8807
Ridge alpha = 3: метрика на округлённых тестовых данных: 0.8807

KNN: метрика на train данных: 0.8590
KNN: метрика на тестовых данных: 0.6501
KNN: метрика на округлённых тестовых данных: 0.6501

Random Forest Classifier: метрика на train данных: 0.8423
Random Forest Classifier: метрика на тестовых данных: 0.7949
Random Forest Classifier: метрика на округлённых тестовых данных: 0.7949



In [18]:
# Словарь моделей регрессии
models_regression = {
    'Linear Regression': LinearRegression(),

    'Lasso Regression': Lasso()
}

for a in np.linspace(0.5, 2.5, 9):
    models_regression[f'Ridge alpha={a}'] = Ridge(a)

In [19]:
models_regression

{'Linear Regression': LinearRegression(),
 'Lasso Regression': Lasso(),
 'Ridge alpha=0.5': Ridge(alpha=0.5),
 'Ridge alpha=0.75': Ridge(alpha=0.75),
 'Ridge alpha=1.0': Ridge(),
 'Ridge alpha=1.25': Ridge(alpha=1.25),
 'Ridge alpha=1.5': Ridge(alpha=1.5),
 'Ridge alpha=1.75': Ridge(alpha=1.75),
 'Ridge alpha=2.0': Ridge(alpha=2.0),
 'Ridge alpha=2.25': Ridge(alpha=2.25),
 'Ridge alpha=2.5': Ridge(alpha=2.5)}

In [20]:
X_train_neg = X_train[12500:]
X_train_pos = X_train[:12500]
y_train_neg = y_train[12500:]
y_train_pos = y_train[:12500]
#y_train_neg.index = y_train_pos.index

X_test_neg = X_test[12500:]  # Теперь это тестовые данные для негативных примеров
X_test_pos = X_test[:12500]   # Теперь это тестовые данные для позитивных примеров
y_test_neg = y_test[12500:]   # Тестовые метки для негативных примеров
y_test_pos = y_test[:12500]    # Тестовые метки для позитивных примеров
#y_test_neg.index = y_test_pos.index

In [21]:
print('MAE на негативных отзывах:')
print()
for model_name, model in models_regression.items():
    test_error(model, model_name, X_test_neg, y_test_neg, X_train_neg, y_train_neg, mean_absolute_error)

MAE на негативных отзывах:

Linear Regression: метрика на train данных: 0.0001
Linear Regression: метрика на тестовых данных: 0.9687
Linear Regression: метрика на округлённых тестовых данных: 0.9396

Lasso Regression: метрика на train данных: 1.0723
Lasso Regression: метрика на тестовых данных: 1.0639
Lasso Regression: метрика на округлённых тестовых данных: 1.0266

Ridge alpha=0.5: метрика на train данных: 0.4079
Ridge alpha=0.5: метрика на тестовых данных: 0.8504
Ridge alpha=0.5: метрика на округлённых тестовых данных: 0.8241

Ridge alpha=0.75: метрика на train данных: 0.4745
Ridge alpha=0.75: метрика на тестовых данных: 0.8422
Ridge alpha=0.75: метрика на округлённых тестовых данных: 0.8182

Ridge alpha=1.0: метрика на train данных: 0.5206
Ridge alpha=1.0: метрика на тестовых данных: 0.8383
Ridge alpha=1.0: метрика на округлённых тестовых данных: 0.8158

Ridge alpha=1.25: метрика на train данных: 0.5552
Ridge alpha=1.25: метрика на тестовых данных: 0.8365
Ridge alpha=1.25: метрика н

In [22]:
print('MAE на позитивных отзывах:')
print()
for model_name, model in models_regression.items():
    test_error(model, model_name, X_test_pos, y_test_pos, X_train_pos, y_train_pos, mean_absolute_error)

MAE на позитивных отзывах:

Linear Regression: метрика на train данных: 0.0000
Linear Regression: метрика на тестовых данных: 1.0267
Linear Regression: метрика на округлённых тестовых данных: 1.0006

Lasso Regression: метрика на train данных: 1.0498
Lasso Regression: метрика на тестовых данных: 1.0428
Lasso Regression: метрика на округлённых тестовых данных: 0.9970

Ridge alpha=0.5: метрика на train данных: 0.4233
Ridge alpha=0.5: метрика на тестовых данных: 0.8794
Ridge alpha=0.5: метрика на округлённых тестовых данных: 0.8506

Ridge alpha=0.75: метрика на train данных: 0.4901
Ridge alpha=0.75: метрика на тестовых данных: 0.8690
Ridge alpha=0.75: метрика на округлённых тестовых данных: 0.8436

Ridge alpha=1.0: метрика на train данных: 0.5362
Ridge alpha=1.0: метрика на тестовых данных: 0.8639
Ridge alpha=1.0: метрика на округлённых тестовых данных: 0.8403

Ridge alpha=1.25: метрика на train данных: 0.5706
Ridge alpha=1.25: метрика на тестовых данных: 0.8614
Ridge alpha=1.25: метрика н

In [24]:
from sklearn.model_selection import GridSearchCV

# Определение модели Ridge
ridge = Ridge()

# Определение параметров для поиска
param_grid = {
    'alpha': np.linspace(0.1, 3, 50)  # Различные значения alpha от 10^(-4) до 10^(4)
}

# Настройка GridSearchCV
grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, 
                           scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)

# Обучение модели
grid_search.fit(X_train_pos, y_train_pos)

# Вывод результатов
print(f"Лучшие гиперпараметры: {grid_search.best_params_}")
print(f"Лучшее значение MSE: {-grid_search.best_score_}")

Лучшие гиперпараметры: {'alpha': 1.1653061224489796}
Лучшее значение MSE: 0.8538755520723635


In [25]:
model_clf = models_classification['Ridge classifier']
model_neg = Ridge(1.75)
model_pos = Ridge(1.75)
test_error(model_pos, 'Positive data', X_test_pos, y_test_pos, X_train_pos, y_train_pos, mean_absolute_error)
test_error(model_neg, 'Negative data', X_test_neg, y_test_neg, X_train_neg, y_train_neg, mean_absolute_error)

Positive: метрика на train данных: 0.6202
Positive: метрика на тестовых данных: 0.8598
Positive: метрика на округлённых тестовых данных: 0.8354

Negative: метрика на train данных: 0.6051
Negative: метрика на тестовых данных: 0.8358
Negative: метрика на округлённых тестовых данных: 0.8110



In [57]:
def process_text(text, preprocess, vectorizer, model_clf, model_neg, model_pos):
    preprop_text = preprocess(text)
    vect_text = vectorizer.transform([preprop_text])
    is_positive = model_clf.predict(vect_text)[0]
    if is_positive:
        return 'positive', round(model_pos.predict(vect_text)[0])
    else:
        
        return 'negative', round(model_neg.predict(vect_text)[0])

In [58]:
process_text("Best movie ever. Heath ledger's work is phenomenal no words......", preprocess_text, vectorizer, model_clf, model_neg, model_pos)

('positive', 10)

In [42]:
def decision(x):
    if x<0.125:
        return 1
    if x<0.25:
        return 2
    if x<0.375:
        return 3
    if x<0.5:
        return 4
    if x<0.625:
        return 7
    if x<0.75:
        return 8
    if x<0.875:
        return 9
    if x<1:
        return 10

In [39]:
def sigmoid(z):
    sigmoid = 1.0/(1.0 + np.exp(-z))
    return sigmoid 


In [45]:
only_clf_pred = list(map(decision, sigmoid(model_clf.decision_function(X_test))))

In [46]:
mean_absolute_error(only_clf_pred, y_test)

1.74964

In [59]:
final_pred = []

for text in df_test.text:
    final_pred.append(process_text(text, preprocess_text, vectorizer, model_clf, model_neg, model_pos)[1])

In [60]:
mean_absolute_error(final_pred, y_test)

1.471

In [52]:
import pickle

In [56]:
# save
with open('model_clf.pkl','wb') as f:
    pickle.dump(model_clf,f)

# save
with open('model_pos.pkl','wb') as f:
    pickle.dump(model_pos,f)


# save
with open('model_neg.pkl','wb') as f:
    pickle.dump(model_neg,f)


with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)